# Projeto Final – Estudo da População


## 1. Importação dos dados




In [1]:
import pandas as pd

caminho_planilha = "CD2022_Populacao_2010_Compatibilizada_20231222.xlsx"
base_censo = pd.read_excel(caminho_planilha, sheet_name="Municípios", header=2)
base_censo = base_censo.dropna(subset=["UF", "NOME DO MUNICÍPIO"]).copy()

print(f"Registros carregados: {len(base_censo):,}")
base_censo.head()


Registros carregados: 5,570


,Unnamed: 0,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,População Município 2010\n(Sinopse),População 2010 (Alterações de Limites até 2022)1,População Censo 2022
0,NaN,RO,11.0,15.0,Alta Floresta D'Oeste,24392.0,24392.0,21494.0
1,NaN,RO,11.0,23.0,Ariquemes,90353.0,90353.0,96833.0
2,NaN,RO,11.0,31.0,Cabixi,6313.0,6313.0,5351.0
3,NaN,RO,11.0,49.0,Cacoal,78574.0,78574.0,86887.0
4,NaN,RO,11.0,56.0,Cerejeiras,17029.0,17029.0,15890.0


## 2. Conferência dos dados



In [2]:
pop_2010 = "População 2010 (Alterações de Limites até 2022)1"
pop_2022 = "População Censo 2022"

base_censo[pop_2010] = pd.to_numeric(base_censo[pop_2010], errors="coerce")
base_censo[pop_2022] = pd.to_numeric(base_censo[pop_2022], errors="coerce")

base_censo["SALDO_POPULACIONAL"] = base_censo[pop_2022].sub(base_censo[pop_2010])

print("Valores ausentes nas populações:")
print(base_censo[[pop_2010, pop_2022]].isna().sum())

base_censo[["UF", "NOME DO MUNICÍPIO", pop_2010, pop_2022, "SALDO_POPULACIONAL"]].head()


Valores ausentes nas populações:
População 2010 (Alterações de Limites até 2022)1    0
População Censo 2022                                0
dtype: int64


,UF,NOME DO MUNICÍPIO,População 2010 (Alterações de Limites até 2022)1,População Censo 2022,SALDO_POPULACIONAL
0,RO,Alta Floresta D'Oeste,24392.0,21494.0,-2898.0
1,RO,Ariquemes,90353.0,96833.0,6480.0
2,RO,Cabixi,6313.0,5351.0,-962.0
3,RO,Cacoal,78574.0,86887.0,8313.0
4,RO,Cerejeiras,17029.0,15890.0,-1139.0


## 3. Análise da variação populacional



In [3]:
qtd_crescimento = (base_censo["SALDO_POPULACIONAL"] > 0).sum()
qtd_reducao = (base_censo["SALDO_POPULACIONAL"] < 0).sum()
qtd_estavel = (base_censo["SALDO_POPULACIONAL"] == 0).sum()

quadro_situacao = pd.DataFrame({
    "Situação": ["Aumento", "Redução", "Sem variação"],
    "Quantidade de municípios": [qtd_crescimento, qtd_reducao, qtd_estavel]
})

quadro_situacao


,Situação,Quantidade de municípios
0,Aumento,3172
1,Redução,2395
2,Sem variação,3


## 4. Consolidação por estado



In [4]:
pop_por_uf = (
    base_censo.groupby("UF", as_index=False)[[pop_2010, pop_2022]]
    .sum()
)

pop_por_uf["SALDO_POPULACIONAL"] = pop_por_uf[pop_2022].sub(pop_por_uf[pop_2010])
pop_por_uf = (
    pop_por_uf.sort_values("SALDO_POPULACIONAL", ascending=False)
    .reset_index(drop=True)
)

pop_por_uf


,UF,População 2010 (Alterações de Limites até 2022)1,População Censo 2022,SALDO_POPULACIONAL
0,SP,41262199.0,44411238.0,3149039.0
1,SC,6248436.0,7610361.0,1361925.0
2,GO,6001789.0,7056495.0,1054706.0
3,PR,10444526.0,11444380.0,999854.0
4,MG,19597330.0,20539989.0,942659.0
5,MT,3035122.0,3658649.0,623527.0
6,PA,7581051.0,8120131.0,539080.0
7,AM,3483985.0,3941613.0,457628.0
8,CE,8451644.0,8794957.0,343313.0
9,ES,3514952.0,3833712.0,318760.0


### Principais resultados por estado

In [5]:
print("5 estados com maior aumento líquido:")
display(pop_por_uf.head(5))

print("5 estados com maior redução líquida:")
display(
    pop_por_uf.nsmallest(5, "SALDO_POPULACIONAL")
)


5 estados com maior aumento líquido:


,UF,População 2010 (Alterações de Limites até 2022)1,População Censo 2022,SALDO_POPULACIONAL
0,SP,41262199.0,44411238.0,3149039.0
1,SC,6248436.0,7610361.0,1361925.0
2,GO,6001789.0,7056495.0,1054706.0
3,PR,10444526.0,11444380.0,999854.0
4,MG,19597330.0,20539989.0,942659.0


5 estados com maior redução líquida:


,UF,População 2010 (Alterações de Limites até 2022)1,População Censo 2022,SALDO_POPULACIONAL
26,AL,3120887.0,3127683.0,6796.0
25,RO,1562409.0,1581196.0,18787.0
24,AP,669526.0,733759.0,64233.0
23,RJ,15989929.0,16055174.0,65245.0
22,AC,733559.0,830018.0,96459.0


## 5. Detalhamento por município



In [6]:
dados_municipais = base_censo[[
    "UF",
    "COD. UF",
    "COD. MUNIC",
    "NOME DO MUNICÍPIO",
    pop_2010,
    pop_2022,
    "SALDO_POPULACIONAL"
]].copy()

dados_municipais = (
    dados_municipais
    .sort_values("SALDO_POPULACIONAL", ascending=False)
    .reset_index(drop=True)
)

dados_municipais.head(20)


,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,População 2010 (Alterações de Limites até 2022)1,População Censo 2022,SALDO_POPULACIONAL
0,AM,13.0,2603.0,Manaus,1802014.0,2063689.0,261675.0
1,DF,53.0,108.0,Brasília,2572159.0,2817381.0,245222.0
2,SP,35.0,50308.0,São Paulo,11253503.0,11451999.0,198496.0
3,SP,35.0,52205.0,Sorocaba,586816.0,723682.0,136866.0
4,GO,52.0,8707.0,Goiânia,1301912.0,1437366.0,135454.0
5,RR,14.0,100.0,Boa Vista,284313.0,413486.0,129173.0
6,SC,42.0,5407.0,Florianópolis,421240.0,537211.0,115971.0
7,PA,15.0,5536.0,Parauapebas,153908.0,267836.0,113928.0
8,MS,50.0,2704.0,Campo Grande,786774.0,898100.0,111326.0
9,PB,25.0,7507.0,João Pessoa,723515.0,833932.0,110417.0


### Maiores aumentos e maiores reduções

In [7]:
print("10 municípios com maior aumento:")
display(dados_municipais.head(10))

print("10 municípios com maior redução:")
display(
    dados_municipais.nsmallest(10, "SALDO_POPULACIONAL")
)


10 municípios com maior aumento:


,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,População 2010 (Alterações de Limites até 2022)1,População Censo 2022,SALDO_POPULACIONAL
0,AM,13.0,2603.0,Manaus,1802014.0,2063689.0,261675.0
1,DF,53.0,108.0,Brasília,2572159.0,2817381.0,245222.0
2,SP,35.0,50308.0,São Paulo,11253503.0,11451999.0,198496.0
3,SP,35.0,52205.0,Sorocaba,586816.0,723682.0,136866.0
4,GO,52.0,8707.0,Goiânia,1301912.0,1437366.0,135454.0
5,RR,14.0,100.0,Boa Vista,284313.0,413486.0,129173.0
6,SC,42.0,5407.0,Florianópolis,421240.0,537211.0,115971.0
7,PA,15.0,5536.0,Parauapebas,153908.0,267836.0,113928.0
8,MS,50.0,2704.0,Campo Grande,786774.0,898100.0,111326.0
9,PB,25.0,7507.0,João Pessoa,723515.0,833932.0,110417.0


10 municípios com maior redução:


,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,População 2010 (Alterações de Limites até 2022)1,População Censo 2022,SALDO_POPULACIONAL
5569,BA,29.0,27408.0,Salvador,2675656.0,2417678.0,-257978.0
5568,RJ,33.0,4557.0,Rio de Janeiro,6320446.0,6211223.0,-109223.0
5567,RJ,33.0,4904.0,São Gonçalo,999728.0,896744.0,-102984.0
5566,PA,15.0,1402.0,Belém,1393399.0,1303403.0,-89996.0
5565,RS,43.0,14902.0,Porto Alegre,1409351.0,1332845.0,-76506.0
5564,MG,31.0,6200.0,Belo Horizonte,2375609.0,2315560.0,-60049.0
5563,RN,24.0,8102.0,Natal,803739.0,751300.0,-52439.0
5562,PE,26.0,11606.0,Recife,1537704.0,1488920.0,-48784.0
5561,RJ,33.0,1702.0,Duque de Caxias,855088.0,808161.0,-46927.0
5560,CE,23.0,4400.0,Fortaleza,2459712.0,2428708.0,-31004.0


## 6. Exportação dos resultados



In [8]:
pop_por_uf.to_csv(
    "populacao_por_estado.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

dados_municipais.to_csv(
    "populacao_por_municipio.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos de população por estado e por município foram atualizados.")


Arquivos de população por estado e por município foram atualizados.
